# Logical Changes: What Changed and Why

This notebook explains *logical* changes introduced in the performance-improvement work, focusing on intent and decision criteria rather than code details.

## 1) Performance goals became explicit
- Added formal acceptance criteria to prevent speed-only regressions.
- Reason: optimization must protect segmentation/counting quality.
- Effect: every candidate can be judged consistently against baseline.


## 2) Benchmarking became reproducible
- Introduced multi-trial benchmark reports with frozen config + environment metadata.
- Reason: single runs are noisy; decisions should use repeated trials and variance.
- Effect: objective baseline-vs-candidate comparisons across speed and quality.


## 3) Bottlenecks became measurable end-to-end
- Added profiling across data load, preprocessing, training fit, model save/load, prediction, and counting.
- Reason: optimize highest wall-clock contributors first.
- Effect: optimization roadmap is guided by measured impact, not guesses.


## 4) Data path moved to throughput-first design
- Shifted training input flow to tf.data with caching, prefetch, and parallel map.
- Reason: Python-heavy preprocessing can starve GPU/CPU compute.
- Effect: steadier step times, better hardware utilization, deterministic runs when seeded.


## 5) Training compute got staged optimization controls
- Added mixed precision toggle, LR schedule refinement, and early stopping/checkpoint strategy.
- Reason: improve time-to-target metrics without sacrificing validation quality.
- Effect: candidate configurations can optimize for speed while preserving quality constraints.


## 6) Model search became architecture-aware
- Added controlled U-Net variants (baseline/light/tiny) with separable-conv options.
- Reason: architecture choices are a major speed/size lever.
- Effect: Pareto selection is possible (latency/training time vs IoU/counting).


## 7) Inference and counting were decoupled
- Benchmarks now separate raw prediction time from post-processing/counting.
- Added fast and high-accuracy post-processing modes.
- Reason: deployment scenarios need clear latency-quality tradeoff options.
- Effect: runtime budget can be matched to use-case constraints.


## 8) Validation and regression guardrails were formalized
- Added tests for metric correctness, data integrity, model shape behavior, and performance aggregation logic.
- Added CI tiers: smoke tests and scheduled benchmark workflow.
- Reason: prevent silent regressions in quality or latency after future changes.
- Effect: repeatable release confidence and safer optimization iterations.


In [ ]:
from pathlib import Path
import json

ROOT = Path('..').resolve()
baseline_path = ROOT / 'benchmark_runs' / 'baseline' / 'benchmark_report.json'
candidate_path = ROOT / 'benchmark_runs' / 'candidate' / 'benchmark_report.json'

def load(p):
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

b = load(baseline_path)
c = load(candidate_path)
if b and c:
    print('Acceptance summary:')
    print(c.get('acceptance', {}))
else:
    print('Run benchmark.py first to view baseline-vs-candidate evidence in this notebook.')
